# Home Exercise 2 on Text Generation
Implement a sequence2sequence to summarize the text. 

Data: [CNN-DailyMail News Text Summarization](https://www.kaggle.com/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail) 

In [1]:
import os, shutil

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gowrishankarp/newspaper-text-summarization-cnn-dailymail")

print("Path to dataset files:", path)

Path to dataset files: /home/dikhang/.cache/kagglehub/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/versions/2


In [3]:
import zipfile
def move_path(src_path: str, dest_dir: str):
    if not os.path.exists(src_path):
        raise FileNotFoundError(f"Source file not found: {src_path}")
    
    os.makedirs(dest_dir, exist_ok=True)
    dst_path = os.path.join(dest_dir, os.path.basename(src_path))
    
    if os.path.exists(dst_path):
        print(f"Destination {dst_path} exists. Overwriting...")
        if os.path.isdir(dst_path):
            shutil.rmtree(dst_path)
        else:
            os.remove(dst_path)

    # Thực hiện di chuyển
    new_path = shutil.move(src_path, dest_dir)
    return new_path

def unzip(path, dest, delete=True):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Zip file does not exist: {path}")
    
    if not zipfile.is_zipfile(path):
        raise zipfile.BadZipFile(f"Not a valid zip file: {path}")
    
    print(f"Extracting {path} to {dest}...")
    with zipfile.ZipFile(path, 'r') as zip_ref:
        zip_ref.extractall(dest)
        print(f"Unzipped successfully into: {dest}")
    
    if delete:
        os.remove(path)
        print(f"Deleted zip file: {path}")
    else:
        print(f"Kept zip file.")
    
    return dest

In [4]:
!ls /home/dikhang/.cache/kagglehub/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/versions/2/cnn_dailymail

test.csv  train.csv  validation.csv


In [5]:
cache_root = "/home/dikhang/.cache/kagglehub/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/versions/2"
source_data_dir = os.path.join(cache_root, "cnn_dailymail")

source_dir = "./data"
# Thư mục đích sẽ chứa file csv
data_dir = os.path.join(source_dir, "cnn_dailymail")

print(f"Source Directory: {source_data_dir}")
print(f"Destination Directory: {data_dir}")

os.makedirs(data_dir, exist_ok=True)

files = ["train.csv", "test.csv", "validation.csv"]

try:
    if not os.path.exists(source_data_dir):
        raise FileNotFoundError(f"Cannot found: {source_data_dir}")

    for file_name in files:
        src_file = os.path.join(source_data_dir, file_name)
        dst_file = os.path.join(data_dir, file_name)
        

        if os.path.exists(src_file):
            shutil.copy2(src_file, dst_file)
            print(f"Đã sao chép: {file_name}")
        else:
            print(f"Không tìm thấy file {file_name} trong nguồn.")
            
except Exception as e:
    print(f"Lỗi nghiêm trọng trong quá trình sao chép: {e}")
    exit(1)

train_path = os.path.join(data_dir, "train.csv")
test_path = os.path.join(data_dir, "test.csv")
val_path = os.path.join(data_dir, "validation.csv") 

print(f"Train path variable: {train_path}")
print(f"Val path variable:   {val_path}")

Source Directory: /home/dikhang/.cache/kagglehub/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/versions/2/cnn_dailymail
Destination Directory: ./data/cnn_dailymail
✔ Đã sao chép: train.csv
✔ Đã sao chép: test.csv
✔ Đã sao chép: validation.csv
Train path variable: ./data/cnn_dailymail/train.csv
Val path variable:   ./data/cnn_dailymail/validation.csv


## Data preparation

### Import libraries

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import spacy
import random
from datetime import datetime
from collections import Counter

print(f"The last time this notebook was run is: {datetime.now().strftime('%H:%M:%S %d/%m/%y')}")

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

The last time this notebook was run is: 17:34:13 14/12/25
Using device: cpu


### Dataset class

In [10]:
class CNNDailyMailDataset(Dataset):
    def __init__(self, csv_path, vocab=None, max_len_article=400, max_len_summary=100):
        self.df = pd.read_csv(csv_path)
        # self.df = self.df.iloc[:1000] 
        self.article_col = 'article'
        self.highlight_col = 'highlights'
        self.max_len_article = max_len_article
        self.max_len_summary = max_len_summary
        
        self.tokenizer = spacy.load("en_core_web_sm")
        
        if vocab is None:
            self.vocab = self.build_vocab()
        else:
            self.vocab = vocab

    def tokenize(self, text):
        return [tok.text.lower() for tok in self.tokenizer(str(text))]

    def build_vocab(self, min_freq=2):
        print("Building Vocabulary...")
        counter = Counter()
        for text in self.df[self.article_col]:
            counter.update(self.tokenize(text))
        for text in self.df[self.highlight_col]:
            counter.update(self.tokenize(text))
            
        vocab = {'<pad>': 0, '<unk>': 1, '<sos>': 2, '<eos>': 3}
        idx = 4
        for word, count in counter.items():
            if count >= min_freq:
                vocab[word] = idx
                idx += 1
        print(f"Vocab size: {len(vocab)}")
        return vocab

    def text_to_indices(self, text, max_len):
        tokens = self.tokenize(text)
        tokens = tokens[:max_len-2]
        indices = [self.vocab.get('<sos>')] + \
                  [self.vocab.get(token, self.vocab.get('<unk>')) for token in tokens] + \
                  [self.vocab.get('<eos>')]
        
        # Padding
        if len(indices) < max_len:
            indices += [self.vocab.get('<pad>')] * (max_len - len(indices))
        return torch.tensor(indices, dtype=torch.long)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        article = self.df.iloc[idx][self.article_col]
        summary = self.df.iloc[idx][self.highlight_col]
        
        src = self.text_to_indices(article, self.max_len_article)
        trg = self.text_to_indices(summary, self.max_len_summary)
        return src, trg

### Loading the data

In [12]:
train_dataset = CNNDailyMailDataset(train_path)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

Building Vocabulary...


KeyboardInterrupt: 